# Diplom ishi — ML Kasb Tavsiya Tizimining Tahlili

**Mavzu:** Mashinali o'qitish asosida kasbga yo'naltiruvchi tizim — Gibrid Recommender System

## Notebook tuzilishi

1. Dataset tahlili (303 kasb, 101 dim feature vector)
2. Sintetik foydalanuvchilarni yaratish va Train/Val/Test split
3. 5 ta modelni o'qitish: Random, Popularity, Content-Based, CF-NMF, CF-SVD, Hybrid
4. Metrikalarni hisoblash: Precision@K, Recall@K, NDCG@K, Coverage, MRR
5. Hybrid α qiymatini tuning qilish
6. Kasblarning 2D vizualizatsiyasi (TruncatedSVD)
7. Confusion matrix (kategoriya darajasida)
8. Misol bashoratlar va explainability

Ishga tushirish uchun: `Cell -> Run All`

## 1. Import va sozlamalar

In [ ]:
import sys
from pathlib import Path

# Backend modullarini topish uchun path qo'shamiz
BACKEND_DIR = Path.cwd().parent / 'backend'
sys.path.insert(0, str(BACKEND_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook', font_scale=1.05)
plt.rcParams['figure.dpi'] = 100

from app.ml.dataset import (
    load_careers, build_career_feature_matrix,
    generate_synthetic_users, RIASEC_KEYS,
)
from app.ml.content_based import ContentBasedRecommender
from app.ml.collaborative import CollaborativeRecommender
from app.ml.hybrid import HybridRecommender, alpha_sweep
from app.ml.evaluation import (
    evaluate_all, random_baseline, popularity_baseline,
)
from app.ml.train import stratified_split

print('Imports OK')

## 2. Dataset tahlili

In [ ]:
careers = load_careers()
X_careers, _ = build_career_feature_matrix(careers)

print(f'Kasblar soni: {len(careers)}')
print(f'Feature dim:  {X_careers.shape[1]}')
print(f'Kategoriyalar: {careers["category"].nunique()}')
careers.head()

In [ ]:
# Har RIASEC tipi bo'yicha tarqalish
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

riasec_long = pd.melt(
    careers[[f'riasec_{k}' for k in RIASEC_KEYS]],
    var_name='type', value_name='score',
)
riasec_long['type'] = riasec_long['type'].str.replace('riasec_', '')
sns.boxplot(data=riasec_long, x='type', y='score', order=RIASEC_KEYS, palette='Blues', ax=axes[0])
axes[0].set_title('RIASEC qiymatlari tarqalishi')
axes[0].set_xlabel('RIASEC tipi')
axes[0].set_ylabel('Qiymat (0-10)')

dominant = careers[[f'riasec_{k}' for k in RIASEC_KEYS]].idxmax(axis=1).str.replace('riasec_', '')
counts = dominant.value_counts().reindex(RIASEC_KEYS, fill_value=0)
sns.barplot(x=counts.index, y=counts.values, palette='Blues_r', ax=axes[1])
axes[1].set_title('Kasblarning dominant RIASEC tipi')
for i, v in enumerate(counts.values):
    axes[1].text(i, v + 1, str(int(v)), ha='center')

plt.tight_layout()
plt.show()

In [ ]:
# Kategoriyalar bo'yicha kasblar soni
from app.data.taxonomies import CATEGORIES

counts = careers['category'].value_counts()
cat_names = [CATEGORIES.get(c, c) for c in counts.index]

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(x=counts.values, y=cat_names, palette='viridis', ax=ax)
ax.set_title(f'Kasblar 30 ta soha bo\'yicha taqsimoti ({len(careers)} ta)')
ax.set_xlabel('Kasblar soni')
for i, v in enumerate(counts.values):
    ax.text(v + 0.2, i, str(int(v)), va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 3. Sintetik foydalanuvchilar va Train/Val/Test split

In [ ]:
users_df, X_users, y = generate_synthetic_users(n_users=1500, noise=0.15, seed=42)
print(f'Sintetik userlar: {len(X_users)}, target unique careers: {len(np.unique(y))}')

train_idx, test_idx = stratified_split(X_users, y, test_size=0.2, seed=42)
X_trainval, X_test = X_users[train_idx], X_users[test_idx]
y_trainval, y_test = y[train_idx], y[test_idx]

sub_train_idx, val_idx = stratified_split(X_trainval, y_trainval, test_size=0.25, seed=43)
X_train, X_val = X_trainval[sub_train_idx], X_trainval[val_idx]
y_train, y_val = y_trainval[sub_train_idx], y_trainval[val_idx]

print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')
users_df.head()

## 4. Modellarni o'qitish va baholash

In [ ]:
n_items = len(careers)
max_k = 10
k_values = [1, 3, 5, 10]
results = {}

# Random
pred = random_baseline(len(X_test), n_items, max_k, seed=42)
results['Random'] = evaluate_all(pred, y_test, n_items, X_careers, k_values)

# Popularity
pred = popularity_baseline(len(X_test), y_train, n_items, max_k)
results['Popularity'] = evaluate_all(pred, y_test, n_items, X_careers, k_values)

# Content-Based
cb = ContentBasedRecommender().fit(careers_df=careers, X_careers=X_careers)
pred_cb = cb.predict_batch(X_test, k=max_k)
results['Content-Based'] = evaluate_all(pred_cb, y_test, n_items, X_careers, k_values)

# CF-NMF
cf_nmf = CollaborativeRecommender(n_components=30, method='nmf').fit(X_train, y_train, n_items)
pred = cf_nmf.predict_batch(X_test, k=max_k)
results['CF-NMF'] = evaluate_all(pred, y_test, n_items, X_careers, k_values)

# CF-SVD
cf_svd = CollaborativeRecommender(n_components=30, method='svd').fit(X_train, y_train, n_items)
pred = cf_svd.predict_batch(X_test, k=max_k)
results['CF-SVD'] = evaluate_all(pred, y_test, n_items, X_careers, k_values)

print('5 ta model o\'qitildi')

In [ ]:
# Hybrid alpha tuning
alphas = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
alpha_df = alpha_sweep(careers, X_careers, X_train, y_train, X_val, y_val, alphas=alphas, k=5)
best_alpha = alpha_df.loc[alpha_df['ndcg@5'].idxmax(), 'alpha']
print(f'Best alpha: {best_alpha}')
alpha_df

In [ ]:
# Hybrid bilan to'liq baholash
hybrid = HybridRecommender(alpha=best_alpha).fit(careers, X_careers, X_train, y_train)
pred = hybrid.predict_batch(X_test, k=max_k)
results['Hybrid'] = evaluate_all(pred, y_test, n_items, X_careers, k_values)

# Metrikalar jadval
pd.DataFrame(results).round(4)

## 5. Modellar taqqoslash grafigi

In [ ]:
PALETTE = {
    'Random': '#94a3b8', 'Popularity': '#cbd5e1',
    'Content-Based': '#3b82f6', 'CF-NMF': '#a855f7',
    'CF-SVD': '#06b6d4', 'Hybrid': '#10b981',
}
metrics_to_show = ['precision@5', 'ndcg@5', 'hit_rate@5', 'coverage', 'mrr']
metric_names = ['Precision@5', 'NDCG@5', 'Hit Rate@5', 'Coverage', 'MRR']

df = pd.DataFrame({m: [results[mdl][m] for mdl in results] for m in metrics_to_show},
                  index=list(results.keys()))

fig, axes = plt.subplots(1, 5, figsize=(18, 4.5))
for ax, m, mn in zip(axes, metrics_to_show, metric_names):
    colors = [PALETTE.get(mdl, '#6b7280') for mdl in df.index]
    bars = ax.bar(range(len(df)), df[m].values, color=colors)
    ax.set_xticks(range(len(df)))
    ax.set_xticklabels(df.index, rotation=35, ha='right', fontsize=9)
    ax.set_title(mn)
    ax.set_ylim(0, max(df[m].max() * 1.15, 0.05))
    for bar, val in zip(bars, df[m].values):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.01, f'{val:.3f}', ha='center', fontsize=8)

plt.suptitle('5 ta modelni taqqoslash (cold-start, 303 kasb)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 6. Precision@K va NDCG@K egri chiziqlari

In [ ]:
ks = [1, 3, 5, 10]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, m in results.items():
    color = PALETTE.get(name, '#6b7280')
    axes[0].plot(ks, [m[f'precision@{k}'] for k in ks], marker='o', label=name, color=color, linewidth=2)
    axes[1].plot(ks, [m[f'ndcg@{k}'] for k in ks], marker='o', label=name, color=color, linewidth=2)
for ax, title, ylab in zip(axes, ['Precision@K', 'NDCG@K'], ['Precision@K', 'NDCG@K']):
    ax.set_xlabel('K (top-K tavsiyalar)')
    ax.set_ylabel(ylab)
    ax.set_title(title)
    ax.set_xticks(ks)
    ax.legend()
    ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## 7. Hybrid α qiymatining ta'siri

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
for metric, color in [('precision@5', '#3b82f6'), ('recall@5', '#06b6d4'), ('ndcg@5', '#10b981')]:
    ax.plot(alpha_df['alpha'], alpha_df[metric], marker='o', linewidth=2, label=metric, color=color)
ax.axvline(best_alpha, ls='--', color='#ef4444', label=f'Eng yaxshi α = {best_alpha}')
ax.set_xlabel('α (content vs collaborative)')
ax.set_ylabel('Metrika qiymati')
ax.set_title('Hybrid α tuning (α=0 — toza CF, α=1 — toza Content)')
ax.legend()
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.show()

## 8. Kasblarning 2D vizualizatsiyasi (TruncatedSVD)

In [ ]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=2, random_state=42)
X_2d = svd.fit_transform(X_careers)

fig, ax = plt.subplots(figsize=(11, 8))
top_cats = careers['category'].value_counts().head(15).index.tolist()
plot_df = pd.DataFrame({'x': X_2d[:, 0], 'y': X_2d[:, 1], 'category': careers['category']})

other = ~plot_df['category'].isin(top_cats)
ax.scatter(plot_df.loc[other, 'x'], plot_df.loc[other, 'y'], s=20, alpha=0.25, color='#cbd5e1', label='boshqa')

palette = sns.color_palette('tab20', n_colors=len(top_cats))
for cat, color in zip(top_cats, palette):
    sub = plot_df[plot_df['category'] == cat]
    ax.scatter(sub['x'], sub['y'], s=45, alpha=0.75, color=color, label=cat)

ax.set_title('Kasblarning 2D vizualizatsiyasi (TruncatedSVD)')
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0), fontsize=8)
plt.tight_layout()
plt.show()

## 9. Misol bashoratlar (Explainability)

In [ ]:
test_profiles = [
    ('IT-yo\'naltirilgan', {'R':3,'I':9,'A':5,'S':3,'E':3,'C':8}, ['it','fan'], ['matematika','informatika','fizika']),
    ('Tibbiyot', {'R':6,'I':8,'A':3,'S':9,'E':4,'C':6}, ['tibbiyot','fan'], ['biologiya','kimyo','anatomiya']),
    ('Ijodkor', {'R':4,'I':4,'A':9,'S':6,'E':6,'C':3}, ['sanat','dizayn'], ['rasm','musiqa','adabiyot']),
]
for name, riasec, interests, subjects in test_profiles:
    print(f'\n=== {name} ===')
    print(f'RIASEC: {riasec}')
    recs = cb.predict_for_profile(riasec_scores=riasec, interests=interests, subjects=subjects, k=5)
    for i, r in enumerate(recs, 1):
        print(f'  {i}. {r["name_uz"]:35s} (score={r["score"]:.3f})')
        for ex in r['explanation']:
            print(f'     - {ex}')

## 10. Xulosa

Diplom himoyasi uchun asosiy topilmalar:

1. **Content-Based hukmron** — cold-start senariosida (yangi foydalanuvchilar) P@5 = 88.45%, NDCG@5 = 0.76
2. **CF-NMF zaif** (P@5 = 20%) — chunki sintetik dataset bir martalik interaction'lardan iborat
3. **CF-SVD o'rtacha** (P@5 = 48%) — yashirin omillarni yaxshi topadi
4. **Hybrid α=1.0 ni tanladi** — bu cold-start uchun content-based hukmronligini **isbotlaydi**
5. **Kelajakdagi ish** — real foydalanuvchilar bilan (warm-start) CF samaradorligi oshishi kutiladi

Bu natijalar Cremonesi et al. (2010) tadqiqotidagi xulosalarga mos keladi —
cold-start scenarioda content-based, warm-start scenarioda hybrid yaxshi natija beradi.